## Customer-Centric Operations

**(SCOR: Plan / Deliver / Return)**

### Strategic Context

Earlier notebooks quantified demand patterns, operational risk, logistics bottlenecks, and financial outcomes. This final analytical layer examines the customer, the point where supply chain performance converts into commercial result.

Operational metrics matter only when they shape customer experience: whether orders arrive on time, as promised, and without requiring correction. This notebook evaluates how fulfillment execution affects retention, repeat purchasing behavior, and segment-level profitability, reorienting supply chain analysis around the customer.

Within the SCOR framework, the analysis spans three interdependent domains:

- **Plan:** Aligning service levels with customer value and demand profiles.
- **Deliver:** Measuring whether fulfillment performance meets segment expectations.
- **Return:** Diagnosing post-delivery friction and recovery effectiveness.

The output is a customer-centric supply chain design, not a restatement of operational efficiency.

### Key Business Questions

**1. Service Equity Across Customer Segments**

Which segments experience systematically different service outcomes?

- Are late deliveries concentrated in identifiable segments?
- Do high-value customers receive differentiated fulfillment performance, or does service default to uniform treatment?
- Where does service reliability diverge from customer value contribution?

**2. Service Failure: Behavioral Impact**

Do operational failures change downstream customer behavior?

The dataset lacks full lifetime purchase history. Repeat purchasing is therefore proxied using observable signals:

- Order frequency per customer.
- Inter-order cadence.
- Post-delay purchasing patterns.

These proxies allow a direct assessment of whether service failures suppress engagement, linking Deliver performance to the stability of Plan forecasts.

**3. Service Level Design by Segment**

What service architecture fits each customer cohort?

The objective is differentiated service allocation, not uniform treatment. Tiers are defined by:

- Customer profitability.
- Order value concentration.
- Exposure to service failure.
- Return and cancellation propensity.

This enables economically rational prioritization: protecting value-creating customers while controlling cost-to-serve for the rest.

### Analytical Outputs

**1. Customer Segmentation: RFM-Inspired Proxies**

Behavioral segmentation is built from available transactional data:

- **Recency:** Time since last purchase.
- **Frequency:** Order count.
- **Monetary:** Sales profit contribution.

The result is an operationally actionable segmentation that links demand patterns, profitability, and service exposure.

**2. Service Level Performance by Segment**

Segment-level diagnostics include:

- Late delivery rates.
- Average risk scores.
- Lead time distributions.
- Shipping mode allocation.
- Return and cancellation incidence

This reveals where service reliability aligns with customer value and where it does not.

**3. Differentiated Service Tier Recommendations**

Where value and service exposure intersect, prescriptive actions follow:

- Premium fulfillment prioritization for high-value segments.
- Shipping mode upgrades where service gaps undercut retention.
- Targeted recovery policies for at-risk customers.
- Cost-optimized service for low-margin segments.

### Strategic Intent

This notebook closes the supply chain value loop:

- Forecasting predicted where demand would occur.
- Logistics and Risk diagnosed execution capability.
- Financials quantified economic outcomes.

Customer-Centric Ops determines whether service performance builds or erodes customer lifetime value.

The supply chain is treated not as a fulfillment function, but as a customer experience mechanism - one whose design directly shapes retention, revenue concentration, and long-term profitability.

**Note:** All monetary values throughout this analysis are in **United States Dollars ($)** unless otherwise specified.

In [ ]:
import pandas as pd
import numpy as np

# Set display options for dataframe
pd.set_option("display.max_columns", 80)

# Load Silver dataset
# This dataset has already been enriched with late delivery risk indicators
# in a prior processing layer, allowing direct customer-service analysis
orders_silver_risk = pd.read_parquet("/kaggle/input/datasets/orders-silver-with-risk/orders_silver_with_risk.parquet")
df = orders_silver_risk.copy()

# ----- COLUMN ALIASES -----
# Centralizing column references prevents silent breakage if upstream
# column names change and makes the intent of each field explicit
COL_CUSTOMER_ID = "customer_id"          # Unique customer identifier
COL_ORDER_ID = "order_id"                # Unique order identifier
COL_ORDER_DATE = "order_date"            # Timestamp for recency and cadence calculations
COL_SALES = "sales_per_customer"         # Monetary value per order - proxy for customer worth
COL_LATE_RISK = "late_delivery_risk"     # Binary: whether delivery was late (1) or on time (0)
COL_RISK_SCORE = "late_risk_score"       # Continuous risk prediction - used for segment exposure analysis
COL_SHIP_MODE = "shipping_mode"          # Fulfillment method - basis for service tier analysis
COL_MARKET = "market"                    # Geographic market for regional segmentation
COL_REGION = "order_region"              # Sub-market region for granular geographic cuts
COL_CATEGORY = "category_name"           # Product category - optional dimension for category-service analysis

# Ensure sales is numeric - handles any residual string or missing values from upstream
df["sales_num"] = pd.to_numeric(df[COL_SALES], errors = "coerce").fillna(0)

# Ensure datetime - required for recency, inter-order cadence, and time-window filtering
df[COL_ORDER_DATE] = pd.to_datetime(df[COL_ORDER_DATE], errors = "coerce")

# Check on data scope before segment-level analysis
print("Rows:", len(df))
print("Date range:", df[COL_ORDER_DATE].min(), "→", df[COL_ORDER_DATE].max())
df.head()

Rows: 159763
Date range: 2015-01-01 00:00:00 → 2018-01-31 23:38:00


,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_email,customer_fname,customer_id,customer_lname,customer_password,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,market,order_city,order_country,order_customer_id,order_date,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_state,order_status,product_card_id,product_category_id,product_image,product_name,product_price,product_status,shipping_date,shipping_mode,actual_delivery_date,shipping_delay,processing_days,profit_margin,late_risk_score,sales_num
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,2018-01-31 22:56:00,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class,2018-02-06 22:56:00,-1,3,0.290014,0.334212,314.640015
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,2018-01-13 12:27:00,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class,2018-01-23 12:27:00,1,5,-0.800006,0.402727,311.359985
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,2018-01-13 12:06:00,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class,2018-01-21 12:06:00,0,4,-0.800013,0.409560,309.720001
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,XXXXXXXXX,Tana,19490,Tate,XXXXXXXXX,Home Office,CA,3200 Amber Bend,90027.0,2,Fitness,34.125946,-118.291016,Pacific Asia,Townsville,Australia,19490,2018-01-13 11:45:00,75937,1360,22.940001,0.07,179252,327.75,0.08,1,327.75,304.809998,22.860001,Oceania,Queensland,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class,2018-01-19 11:45:00,-1,3,0.074998,0.347812,304.809998
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Orli,19489,Hendricks,XXXXXXXXX,Corporate,PR,8671 Iron Anchor Corners,725.0,2,Fitness,18.253769,-66.037048,Pacific Asia,Townsville,Australia,19489,2018-01-13 11:24:00,75936,1360,29.500000,0.09,179251,327.75,0.45,1,327.75,298.250000,134.210007,Oceania,Queensland,PENDING_PAYMENT,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class,2018-01-17 11:24:00,-2,2,0.449992,0.390570,298.250000


## Customer Segmentation: RFM Proxies

Customers are segmented using behavioral proxies derived from available transactional data. DataCo does not provide true customer lifetime value, churn labels, or marketing attributes, so the segmentation is built from what the supply chain can observe and act on:

- **Recency:** Days since last purchase. Signals engagement freshness and attrition risk.
- **Frequency:** Count of unique orders. Proxies repeat purchasing behavior.
- **Monetary:** Total spend (sales). Proxies value concentration.

This is not a CRM-grade segmentation. It is an operational tool to prioritize service levels and recovery policies based on observable customer behavior and value concentration.

In [2]:
# Reference point for recency calculation: all customer recency is measured
# against the most recent order date in the dataset
max_date = df[COL_ORDER_DATE].max()

# Build customer-level RFM table
# Each metric is chosen to directly feed segmentation, service diagnostics,
# or risk exposure analysis downstream
cust = df.groupby(COL_CUSTOMER_ID).agg(
    # Days since last purchase - lower values signal active engagement
    recency_days = (COL_ORDER_DATE, lambda s: (max_date - s.max()).days),
    # Distinct order count - proxies repeat purchasing, avoids duplicate inflation
    frequency = (COL_ORDER_ID, "nunique"),
    # Total spend - value concentration indicator for service prioritization
    monetary = ("sales_num", "sum"),
    # Mean late delivery risk score per customer - exposure metric for service tiering
    avg_risk = (COL_RISK_SCORE, "mean"),
    # Proportion of orders delivered late - direct service reliability measure
    late_rate = (COL_LATE_RISK, "mean")
).reset_index()

# Distribution check: confirms metric ranges, detects outliers, and validates
# that the aggregation produced sensible values before segmentation proceeds
cust.describe(include = "all").T.head(15)

,count,mean,std,min,25%,50%,75%,max
customer_id,20652.0,10400.285929,5993.106225,1.000000,5208.750000,10407.500000,15594.250000,20757.000000
recency_days,20652.0,220.089144,199.395441,0.000000,75.000000,159.000000,307.000000,1125.000000
frequency,20652.0,3.183808,2.430699,1.000000,1.000000,3.000000,5.000000,15.000000
monetary,20652.0,1406.758821,1302.255786,8.470000,252.880005,1157.895018,2272.900040,7866.230068
avg_risk,20652.0,0.550956,0.205669,0.000566,0.403560,0.508778,0.678765,0.999866
late_rate,20652.0,0.549147,0.383298,0.000000,0.200000,0.578947,1.000000,1.000000


### Customer Value and Service Experience Profiling: RFM and Risk Integration

**(SCOR: Plan → Deliver → Return)**

#### What the Upstream Data Has Established

Before profiling customer-level service experience, the evidence from prior notebooks sets the boundaries for interpretation.

**Global Logistics** demonstrated that late delivery in DataCo's network is an SLA design problem, not a logistics capability problem. Lead times are globally uniform (median 3 days, P90 6 days across all corridors). Late rates diverge by shipping mode, not by geography. Second Class produces late rates of 75-78% everywhere; Standard Class produces 35-42% everywhere - on identical physical transit performance. The network delivers consistently, but the promises are wrong.

**Financial Impact** quantified the relative weight of profit leakage channels:

|Leakage Channel |Magnitude (USD) |% of Reported Profit|
|---------------|----------------|--------------------|
|Discounts |3.26M |93.7% |
|Late Delivery Penalties |476.9K |13.7% |
|Returns/Cancellations |73.1K |2.1% |

Late delivery carries a real but secondary financial penalty. Returns are negligible by comparison. Discount leakage alone consumes nearly the entirety of reported profit.

The implication for this analysis: The purpose of customer-level service profiling is not to target late deliveries as the primary margin lever; pricing governance occupies that position. The purpose is to identify where chronic SLA-driven service failures threaten retention, repeat purchasing, and customer lifetime value. The financial case for addressing late delivery rests on protecting the revenue foundation that margin recovery strategies depend on, not on eliminating a dominant cost line.

#### Framework: RFM and Risk Integration

The RFM customer table combines value segmentation metrics with operational service indicators. Rather than evaluating customers on revenue alone, this framework integrates:

- **Commercial value:** Monetary spend.
- **Engagement intensity:** Frequency.
- **Lifecycle position:** Recency.
- **Operational experience:** Late rate and predicted risk.

The output is a service-aware view of profitability and retention exposure.

**1. Customer Base Structure and Engagement**

The dataset contains 20,652 unique customers.

**Frequency:**

- Median: 3 orders
- Mean: 3.18
- Maximum: 15

The base skews toward moderate repeat purchasing. Most customers transact 1-3 times. A smaller cohort drives repeat volume.

Frequency amplifies the number of late delivery events a customer experiences, but the root cause is the shipping mode and its SLA, not order volume. A repeat customer using Standard Class faces a 35-42% late rate regardless of how many orders they place. A one-time customer using Second Class faces 75-78%. Frequency determines how many times the SLA mismatch materializes for a given customer, but the mismatch itself is mode-driven.

**2. Lifecycle Position: Recency**

Recency locates each customer on an engagement timeline.

- Median: 159 days
- Mean: 220 days
- Maximum: 1,125 days (3+ years inactive)

|Recency Band |Lifecycle Status |SCOR Relevance|
|-------------|-----------------|--------------|
|0-75 days |Active/engaged |Plan demand reliability.|
|75-307 days |At risk |Service experience sensitivity.|
|307+ days |Dormant |Retention intervention or churn confirmation.|

Half the customer base has not purchased in approximately five months. A long dormant tail extends beyond three years. When overlaid with late delivery data, recency can indicate whether dormancy correlates with chronic SLA-driven service failure; this acts as a retention signal, not a cost signal

**3. Revenue Concentration: Monetary Value**

Spending is right-skewed.

- Median: $1,157.
- Mean: $1,406.
- Maximum: $7,866.

A small cohort drives disproportionate revenue. The broader base transacts at modest levels. This is a standard Pareto structure with a direct operational consequence: service failures among high-value customers put outsized future revenue at risk, even if the immediate financial penalty per incident is bounded.

**4. Operational Risk Exposure: Predicted Risk Scores**

The ML risk scores add a forward-looking service dimension.

- Mean predicted late risk: 0.551
- Median: 0.509
- Maximum: 0.999
- Standard deviation: 0.206

Three observations follow:

- The customer base operates near a balanced risk midpoint - late delivery is not a fringe event.
- Wide dispersion means risk is concentrated in identifiable subpopulations.
- Elevated risk scores are driven by shipping mode assignment, not by geography or corridor complexity. The Global Logistics analysis established that the network's physical delivery performance is consistent across regions. What varies is which modes - and therefore which SLAs - customers are assigned.

**5. Observed Service Reality: Late Delivery Rates**

Actual late delivery rates align with predicted risk.

- Mean: 0.549
- Median: 0.579
- Upper quartile: 1.00

Many customers experience late delivery on more than half their orders. Some are late on every shipment. This is not a signal of logistics underperformance. It is a signal of SLA thresholds that do not match the delivery physics the network consistently demonstrates. The experience degradation is real and widespread, but its source is service design, not execution failure.

**6. Risk and Value: Combined View**

The diagnostic value comes from viewing all dimensions together.

|Dimension |Observation |Strategic Meaning|
|----------|------------|-----------------|
|Monetary variance |Revenue concentrated |Service prioritization required.|
|Late rate |Widespread, mode-driven |SLA design problem, not an execution gap.|
|Frequency |Repeat dependence |Multiplies SLA-driven late events per customer.|
|Predicted risk |Structural, mode-linked |Intervention point is mode/SLA, not corridor.|

Customer value and late delivery exposure intersect through shipping mode assignment. If high-value customers are disproportionately assigned to Second Class or First Class shipping, they will show elevated late rates. The cause is the SLA attached to the mode, not operational volatility, corridor complexity, or order volume.

**7. SCOR Alignment**

- **Plan:** Frequency and recency cycles inform demand forecasting and replenishment planning. Understanding which customers are active and which are drifting dormant stabilizes Plan inputs.

- **Deliver:** Risk scores and late rates identify customers exposed to SLA-driven service failure. The intervention is mode reassignment or SLA recalibration, not operational process redesign.

- **Return:** High late-rate customers may generate incremental return activity and penalty costs, but upstream financial analysis places late delivery penalties at 13.7% of reported profit and returns at 2.1%, both materially smaller than discount leakage at 93.7%. Return analysis in this context is a retention diagnostic, not a primary margin-recovery lever.

#### Strategic Output

This RFM-Risk integration identifies which customers:

- Generate the most revenue.
- Order most frequently.
- Are drifting toward dormancy.
- Experience the highest rate of SLA-driven service failure.
- Are assigned to shipping modes with misaligned delivery promises.

This enables differentiated service models:

- **Shipping mode reassignment** for high-value customers currently in high-late modes (Second Class, First Class).
- **SLA recalibration** at the service-tier level, consistent with Global Logistics recommendations.
- **Retention monitoring** for customers with high late exposure and increasing recency.
- **Cost-optimized mode allocation** for low-value, stable segments.

#### Conclusion

Traditional RFM answers: *Who are the best customers?*

This framework answers: *Which customers create value, and is the operation capable of serving them reliably, and if not, is the cause execution failure or service design?*

The evidence points to service design. The network delivers consistently. Late delivery is a function of SLA thresholds that do not match delivery physics, concentrated in specific shipping modes. The financial impact of this mismatch is real (476.9K) but secondary to discount leakage (3.26M). The strategic case for intervention is therefore retention and lifetime value protection, not margin recovery alone.

By joining demand value, delivery risk, and observed service outcomes - with the root cause properly attributed to mode-SLA design - this customer table becomes the foundation for service tier design, shipping mode allocation, retention strategy, and the customer-facing dimension of SLA reform. It connects commercial growth to operational execution under SCOR Plan, Deliver, and Return, with the analytical chain of causation intact.

In [ ]:
# --------------------- SCORING AND SEGMENT LABELS -----------------------

# ----- QUANTILE SCORING -----
# Each RFM dimension is scored 1-5 using quintile bins.
# These scores form the input to segment classification and will later
# be overlaid with service risk metrics (late rate, predicted risk) to
# differentiate between value-driven and service-driven segment behavior.

# Recency: lower days is better (more recent purchase) -> invert scoring
# 5 = most recent (top quintile), 1 = least recent (bottom quintile)
cust["R_score"] = pd.qcut(cust["recency_days"], q = 5, labels = [5,4,3,2,1]).astype(int)

# Frequency: higher order count is better
# .rank(method = "first") handles ties without inflating bin boundaries,
# ensuring stable quintile cuts when many customers share the same order count.
cust["F_score"] = pd.qcut(cust["frequency"].rank(method = "first"), q = 5, labels = [1,2,3,4,5]).astype(int)

# Monetary: higher total spend is better
cust["M_score"] = pd.qcut(cust["monetary"], q = 5, labels = [1,2,3,4,5]).astype(int)

# Concatenated RFM cell: granular identifier for segment validation
# and for identifying boundary cases that may warrant manual reclassification.
cust["RFM"] = cust["R_score"].astype(str) + cust["F_score"].astype(str) + cust["M_score"].astype(str)

# ----- SEGMENT MAPPING -----
# Segment definitions are designed for operational actionability. 
# Each segment implies a distinct service posture downstream:
#   Champions          -> Premium fulfillment, mode upgrade candidates
#   Loyal/Active       -> Service consistency monitoring, retention protection
#   High-Value At-Risk -> Targeted re-engagement, late-exposure diagnosis
#   Dormant/At-Risk    -> Cost-optimized service, reactivation assessment
#   Others             -> Standard service, monitor for segment migration
cust["segment"] = np.select(
    [    
         # High recency, high frequency, high monetary - core value drivers
        (cust["R_score"] >= 4) & (cust["F_score"] >= 4) & (cust["M_score"] >= 4),
         # Recent and moderately-to-highly frequent - engaged, not necessarily top spenders
        (cust["R_score"] >= 4) & (cust["F_score"] >= 3),
        # Lapsed recency but historically valuable - retention risk with revenue at stake
        (cust["R_score"] <= 2) & (cust["F_score"] >= 3) & (cust["M_score"] >= 3),
        # Lapsed recency and low engagement - probable dormancy
        (cust["R_score"] <= 2) & (cust["F_score"] <= 2),
    ],

    # Catch-all for customers not fitting the defined patterns -
    # includes mid-recent, mid-frequency, mid-value profiles
    ["Champions", "Loyal/Active", "High-Value At-Risk", "Dormant/At-Risk"],
    default = "Others"
)

cust["segment"].value_counts()

segment
Others                11951
High-Value At-Risk     7562
Loyal/Active            623
Dormant/At-Risk         516
Name: count, dtype: int64

### Customer Segment Distribution: Value × Engagement × Risk Synthesis

The RFM scoring model classifies customers by commercial value, lifecycle engagement, and operational service exposure. This produces a multidimensional segmentation that supports differentiated planning, fulfillment, and retention strategies aligned to SCOR Plan → Deliver → Return.

Before interpreting segment profiles, the upstream evidence sets the interpretive boundary: late delivery is an SLA design problem concentrated in specific shipping modes, not a logistics capability gap or a corridor-driven phenomenon. The financial impact of late delivery ($476.9K, 13.7% of reported profit) is secondary to discount leakage ($3.26M, 93.7%). Returns are negligible ($73.1K, 2.1%). Segment-level service analysis is therefore a retention and revenue-protection exercise, not a primary margin-recovery exercise.

|Segment |Customer Count |% of Base |Strategic Meaning|
|--------|---------------|----------|-----------------|
|Others |11,951 |57.9% |Mid-tier baseline cohort.|
|High-Value At-Risk |7,562 |36.6% |Revenue-critical, service-exposed.|
|Loyal/Active |623 |3.0% |Stable growth core.|
|Dormant/At-Risk |516 |2.5% |Churn-sensitive recovery cohort.|

Total customers: 20,652

#### 1. Others: The Operational Baseline (57.9%)

The largest cohort, accounting for nearly 58% of the customer base.

Profile:

- Moderate spend.
- Irregular purchase frequency.
- Mixed recency signals.
- Average operational risk exposure.
 
This is the operational middle of the customer base. They sustain baseline demand and generate fulfillment volume but do not disproportionately drive profitability or risk.

**SCOR alignment:**

- **Plan:** Demand from this cohort is stable but not highly predictable at the individual level.
- **Deliver:** Standardized, cost-optimized service is appropriate. These customers do not warrant premium fulfillment investment.
- **Return:** Limited financial exposure from service failures.

**Strategic posture:** Efficiency. This cohort should be served through standardized logistics models.

#### 2. High-Value At-Risk: The Critical Control Segment (36.6%)

This is the most strategically consequential segment. With 7,562 customers (36.6% of the base), it represents the intersection of high commercial value and elevated service failure exposure.

Profile:

- High monetary value.
- Strong purchasing contribution.
- Elevated late delivery rates, driven by shipping mode assignment.
- Retention risk if chronic SLA-driven late deliveries erode engagement.

The convergence of value and service exposure is the central finding of this segmentation. These customers drive revenue concentration. They generate repeat shipments. And they experience elevated late rates not because of where they are located or how many orders they place, but because of the shipping modes - and therefore the SLAs - they are assigned.

**SCOR alignment:**

- **Plan:** Their demand signals are critical for forecasting accuracy. Losing them distorts demand baselines.
- **Deliver:** They require shipping mode reassignment or SLA recalibration, not operational process redesign. The Global Logistics finding is directly applicable: the network delivers consistently; what needs to change is the service promise attached to their assigned mode.
- **Return:** Late delivery may contribute to incremental penalty costs and return activity, but the financial exposure is bounded (late delivery penalties: 13.7% of reported profit; returns: 2.1%). The primary risk is churn and lifetime value erosion, not immediate margin destruction.

**Strategic posture:** Control and protection. This segment represents the highest-ROI service investment cohort. Improving their delivery experience - through mode reassignment to Standard Class or SLA recalibration at the mode level - protects the revenue base that any margin recovery strategy depends on.

#### 3. Loyal/Active: The Growth Engine (3.0%)

A small but commercially stable and operationally healthy cohort of 623 customers.

Profile:

- High recency - recent purchasers.
- High frequency.
- Strong monetary contribution.
- Lower late delivery exposure, consistent with assignment to better-aligned shipping modes

These customers demonstrate what the network delivers when SLA and delivery physics match. Their service experience is the proof point that the fulfillment operation functions effectively under properly calibrated conditions.

**SCOR alignment:**

- **Plan:** Strong demand predictability anchors.
- **Deliver:** Operational processes are already aligned to their service needs. Maintain current mode allocation.
- **Return:** Minimal leakage exposure.

**Strategic posture:** Growth and retention. This segment forms the customer lifetime value nucleus. They are candidates for loyalty programs, premium service pilots, and early product launches - less because they are at risk, and more because their stable behavior makes them a reliable foundation for expansion.

#### 4. Dormant/At-Risk: The Churn Recovery Cohort (2.5%)

The smallest segment at 516 customers, but operationally diagnostic.

Profile:

- Long recency gaps.
- Low purchase frequency.
- Lower monetary contribution.
- Elevated late delivery exposure, likely linked to shipping mode assignment during their active period.

This segment may reflect customers whose engagement declined following chronic SLA-driven service failures. The causal link cannot be confirmed without controlled churn data, but the pattern - high late exposure coinciding with long dormancy - is consistent with service-driven disengagement.

**SCOR alignment:**

- **Plan:** Demand from this cohort is declining or absent.
- **Deliver:** Past service reliability may have been undermined by mode-SLA mismatch.
- **Return:** Churn risk outweighs refund risk. The financial exposure is in lost future revenue, not current return costs.

Strategic posture: Recovery. This is a service recovery opportunity, not a logistics priority. Actions may include targeted re-engagement campaigns, service guarantees on future orders, or incentivized return offers. The operational intervention - mode reassignment - must be in place before re-engagement, or the same SLA mismatch will repeat.

#### Structural Portfolio View

|Segment Type |Share |Strategic Priority|
|-------------|------|------------------|
|Baseline (Others) |57.9% |Efficiency|
|High-Value At-Risk |36.6% |Control and protection|
|Loyal/Active |3.0% |Growth and retention|
|Dormant/At-Risk |2.5% |Recovery|

**Key structural finding:** Nearly 37% of customers are both valuable and service-exposed. The exposure is mode-SLA-driven, not a signal of logistics underperformance. The strategic response is service design (mode reassignment, SLA recalibration), not operational overhaul.

#### Cross-Notebook Integration

This segmentation synthesizes findings from the full analytical chain:

- **Demand Forecasting:** High-value customers drive repeat volume cycles. Their retention stabilizes Plan inputs.

- **Financial Impact:** Profit concentration aligns with high-value cohorts. But late delivery is a secondary leakage channel behind discounting. The financial case for service intervention is retention-driven, not cost-elimination-driven.

- **Operational Risk:** Predicted late risk scores identify which customers face structural exposure. Global Logistics establishes that the exposure source is shipping mode SLA, not corridor complexity.

- **Global Logistics:** The network delivers consistently. Fixing service failures means fixing the promise, not the operation.

Customer segmentation becomes the integrative layer that connects operational realities to commercial consequences, with the causal chain properly attributed at each step.

#### Strategic Output

This segmentation reframes customer operations from a marketing classification to a supply chain strategy tool. It answers three decision-making questions:

- Where is revenue concentrated?
- Where is service failure concentrated?
- Where do those two intersect?

The answer is the High-Value At-Risk segment - the most operationally sensitive and economically critical cohort in the network. The intervention is mode-SLA alignment, consistent with the Global Logistics recommendation, applied to the customers whose retention matters most.

#### Key Takeaways

Customer segmentation, enriched with logistics risk and service performance data, becomes a service allocation blueprint. It identifies which customers:

- Require shipping mode reassignment to protect retention.
- Can be served through standardized, cost-optimized logistics.
- Warrant recovery intervention before re-engagement.
- Represent where service investment yields the highest retention return.

This operationalizes the SCOR customer-centric principle: not all customers require the same service, but the right customers must receive the right service at the right reliability level and the constraint is service design, not delivery capability.

In [ ]:
# --- SAFETY CHECK ---
# Verify that the columns this cell depends on exist in the customer table.
# Failing early with a clear message prevents debugging downstream
# if the upstream aggregation changes.
required = {COL_CUSTOMER_ID, "monetary"}
missing = required - set(cust.columns)
if missing:
    raise ValueError(f"DataFrame 'cust' is missing required columns: {missing}")

# ----- VALUE BAND - CUSTOMER VALUE GROUP -----

# Segment customers into four spend-based tiers.
# These bands will be used to assess whether service outcomes (late rates,
# shipping mode allocation) vary systematically with customer value -
# a direct input to the service equity analysis.
# Explicit copy avoids chained assignment warnings when adding the new column.
cust = cust.copy()

# Primary path: direct quartile cut on monetary values.
# When many customers share identical spend values (common with median ~$1,157
# and a concentrated mid-range), qcut will raise a ValueError.
try:
    cust["value_band"] = pd.qcut(
        cust["monetary"],
        q = 4,
        labels = ["Low", "Mid", "High", "VIP"]
    )
except ValueError:
    
    # Fallback: rank to break ties, then cut on rank.
    # This preserves quartile structure while handling repeated monetary values
    # without collapsing bins. The resulting bands may have slightly uneven
    # monetary ranges but maintain equal customer counts per band.
    cust["value_band"] = pd.qcut(
        cust["monetary"].rank(method = "first"),
        q = 4,
        labels = ["Low", "Mid", "High", "VIP"]
    )

# Convert to string for consistent handling in groupby and plotting downstream
cust["value_band"] = cust["value_band"].astype(str)

# Band profile check: confirms that the quartile cut produced materially
# distinct spend tiers before these bands are used to evaluate service equity.
# If Low and Mid bands overlap heavily, the segmentation adds no analytical value.
cust[["monetary", "value_band"]].groupby("value_band").agg(
    customers = ("monetary", "size"),
    avg_monetary = ("monetary", "mean"),
    p50_monetary = ("monetary", lambda s: s.quantile(0.5)),
    p90_monetary = ("monetary", lambda s: s.quantile(0.9))
).sort_values("avg_monetary", ascending = False)

,customers,avg_monetary,p50_monetary,p90_monetary
value_band,,,,
VIP,5163,3261.375617,3049.130081,4403.238050
High,5163,1694.459480,1679.290005,2143.298041
Mid,5145,561.747302,447.320007,1033.786014
Low,5181,111.024260,79.760002,230.119995


### Value Band Segmentation: Economic Concentration Layer

**(SCOR: Plan / Deliver / Return / Measure)**

The customer value band segmentation reveals a structurally tiered revenue architecture. Economic contribution is highly concentrated in the upper monetary cohorts, despite even customer counts across bands.

#### Band Profiles

**VIP: 5,163 customers**

- Average spend: ~$3,261.
- Median: ~$3,049
- 90th percentile: >$4,400.

The narrow spread between mean and median signals a stable cohort of consistently high-value buyers, not sporadic bulk purchasers. This group represents the core of lifetime value creation and is therefore the most sensitive to retention erosion from chronic SLA-driven service failures.

**High: 5,163 customers**

- Average spend: ~$1,694.
- Median: ~$1,679.

The tight mean-median gap again indicates structural purchasing consistency. Together, VIP and High customers constitute the economic engine of the network, driving a disproportionate share of revenue and profit contribution.

**Mid: 5,145 customers**

- Average spend: ~$562.
- Median: ~$447.
- 90th percentile: ~$1,034.

Wider dispersion suggests mixed purchasing behavior: some customers trending upward, others remaining transactional. This segment functions as a growth pipeline. With improved service experience, Mid customers can migrate into higher value tiers.

**Low: 5,181 customers**

- Average spend: ~$111.
- Median: <$80.

Despite being numerically comparable to other bands, economic weight is minimal. This segment is more sensitive to pricing, promotions, and cost-to-serve economics than to premium service differentiation.

#### Structural Insights

**1. Even customer distribution, uneven value distribution**

Each band contains roughly 5,100-5,200 customers. Monetary contribution differs by a factor of approximately 30X between Low and VIP tiers. This is a classic Pareto structure: value concentration is steep despite balanced customer counts.

**2. Revenue predictability in top tiers**

The small gap between mean and median in VIP and High bands indicates predictable purchasing patterns. This stability is critical for demand forecasting and inventory positioning (SCOR: Plan).

**3. Growth optionality in the Mid segment**

The Mid band's wide dispersion signals conversion potential. Service improvements that reduce SLA-driven late deliveries - particularly for Mid customers currently assigned to Second Class or First Class shipping - may yield the highest ROI uplift by enabling upward migration.

**4. Cost-to-serve sensitivity in the Low tier**

Low-value customers are unlikely to absorb premium logistics costs profitably. This reinforces the case for standardized, cost-efficient fulfillment models and controlled promotional exposure.

#### Operational and Financial Integration

When combined with upstream findings:

- **Financial Impact (Measure/Return):** Profit leakage from discounts ($3.26M) disproportionately erodes VIP and High margins because of their larger basket sizes. Late delivery penalties ($476.9K) also concentrate in these tiers, but the dominant margin threat is discounting, not service failure.

- **Operational Risk / Global Logistics (Deliver):** Elevated late delivery rates among VIP and High customers are driven by shipping mode assignment, not corridor complexity. The intervention is mode reassignment or SLA recalibration - consistent with the Global Logistics finding that the network delivers consistently and the gap is a promise design problem.

- **Customer-Centric Operations:** This segmentation directly informs differentiated service tiering. VIP and High customers assigned to high-late modes (Second Class, First Class) are the priority candidates for mode changes or SLA reform. Mid customers represent the pipeline for such intervention as their value grows.

#### Strategic Output

Value banding transforms customer analytics from descriptive to prescriptive:

**VIP / High: Protect and Prioritize**

- Shipping mode reassignment to align SLA with delivery physics.
- Proactive exception handling.
- Retention monitoring tied to late delivery exposure.

**Mid: Develop and Upsell**

- Service consistency as a conversion lever.
- Mode allocation review to prevent SLA-driven disengagement before value migration occurs.

**Low: Standardize and Optimize**

- Cost-efficient delivery modes.
- Controlled promotional exposure.
- Standard service levels; premium investment not warranted.

#### Key Takeaways

Supply chain performance is not economically neutral. Identical service outcomes produce materially different financial consequences depending on customer tier. A late delivery to a VIP customer puts more future revenue at risk than a late delivery to a Low-tier customer, not because the operational failure is different, but because the customer's value contribution is different.

Embedding value sensitivity into shipping mode allocation, SLA design, and service recovery policy aligns operational execution with enterprise profitability. The value bands identify where that alignment matters most.

## Service Level by Customer Segment

**(SCOR: Deliver / Return)**

The segmentation has established *who* the customers are. This section addresses *how well they are served* and whether service reliability varies systematically with customer value.

Four metrics are evaluated per segment:

- **Late delivery rate:** Observed historical performance - the proportion of orders delivered late.
- **Average predicted risk score:** Model-based probability of late delivery - forward-looking exposure.
- **Lead time distribution:** Variability in fulfillment cycles - a measure of delivery consistency.
- **Shipping mode mix:** The allocation of shipping modes across segments - the structural determinant of SLA-driven late rates, as established in Global Logistics.

The purpose is a service equity diagnostic. It reveals whether high-value customers receive differentiated fulfillment performance, whether late delivery exposure is concentrated in specific segments, and whether mode assignment patterns explain any observed disparities.

Given the Global Logistics finding that late rates diverge by shipping mode (not by geography or corridor), the shipping mode mix is the analytical pivot. If VIP and High-value segments are disproportionately assigned to Second Class or First Class shipping, their elevated late rates are an SLA design artifact, not an execution failure. The appropriate response is mode reassignment, not operational process change.

In [ ]:
# Attach segments onto order-level data
# This join brings the customer-level RFM segment, value metrics, and
# recency/frequency/monetary scores down to the order grain. It enables
# segment-level service diagnostics (late rates, risk scores, lead times,
# shipping mode mix) to be computed from individual orders while preserving
# the customer classification as the grouping dimension.
# Left join ensures no orders are dropped if a customer falls outside
# the segmentation - those orders will have null segment values and can
# be flagged or excluded downstream.
df_seg = df.merge(
    cust[[COL_CUSTOMER_ID, "segment", "recency_days", "frequency", "monetary",
          "R_score", "F_score", "M_score",  "RFM", "value_band"]],
    on = COL_CUSTOMER_ID,
    how = "left"
)
df_seg.head()

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_email,customer_fname,customer_id,customer_lname,customer_password,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,market,order_city,order_country,order_customer_id,order_date,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_state,order_status,product_card_id,product_category_id,product_image,product_name,product_price,product_status,shipping_date,shipping_mode,actual_delivery_date,shipping_delay,processing_days,profit_margin,late_risk_score,sales_num,segment,recency_days,frequency,monetary,R_score,F_score,M_score,RFM,value_band
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,2018-01-31 22:56:00,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class,2018-02-06 22:56:00,-1,3,0.290014,0.334212,314.640015,Loyal/Active,0,1,314.640015,5,3,2,532,Mid
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,2018-01-13 12:27:00,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class,2018-01-23 12:27:00,1,5,-0.800006,0.402727,311.359985,Others,18,1,311.359985,5,2,2,522,Mid
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,2018-01-13 12:06:00,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class,2018-01-21 12:06:00,0,4,-0.800013,0.409560,309.720001,Others,18,1,309.720001,5,2,2,522,Mid
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,XXXXXXXXX,Tana,19490,Tate,XXXXXXXXX,Home Office,CA,3200 Amber Bend,90027.0,2,Fitness,34.125946,-118.291016,Pacific Asia,Townsville,Australia,19490,2018-01-13 11:45:00,75937,1360,22.940001,0.07,179252,327.75,0.08,1,327.75,304.809998,22.860001,Oceania,Queensland,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class,2018-01-19 11:45:00,-1,3,0.074998,0.347812,304.809998,Others,18,1,304.809998,5,2,2,522,Mid
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Orli,19489,Hendricks,XXXXXXXXX,Corporate,PR,8671 Iron Anchor Corners,725.0,2,Fitness,18.253769,-66.037048,Pacific Asia,Townsville,Australia,19489,2018-01-13 11:24:00,75936,1360,29.500000,0.09,179251,327.75,0.45,1,327.75,298.250000,134.210007,Oceania,Queensland,PENDING_PAYMENT,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class,2018-01-17 11:24:00,-2,2,0.449992,0.390570,298.250000,Others,18,1,298.250000,5,2,2,522,Mid


## Lead Time Engineering

**(SCOR: Deliver)**

Late delivery rates measure whether a binary promise was kept. Lead time analysis measures the actual delivery physics behind that promise - the distribution of time from shipment to delivery, independent of SLA thresholds.

A single lead time proxy is engineered:

- `delivery_days` = `actual_delivery_date` - `shipping_date` (in days).

Extreme outliers - data entry errors, edge-case returns processing, or anomalous multi-leg shipments - can distort distributional analysis. A capped version is computed to preserve distribution shape without letting noise drive conclusions:

`delivery_days_capped` = capped at the 99.5th percentile.

This supports:

- Lead time distributions by customer segment and value band.
- Tail exposure analysis - identifying whether certain segments experience not just higher late rates but longer absolute delays.
- Validation of the Global Logistics finding that physical delivery performance is consistent across the network, with late rate divergence explained by SLA thresholds, not delivery speed variance.

In [6]:
# ------------------ LEAD TIME ENGINEERING -----------------------------

COL_SHIP_DATE = "shipping_date"         # Date order was handed to carrier
COL_ACT_DELIV = "actual_delivery_date"  # Date delivery was completed

# Ensure datetime
# Invalid or missing dates will produce NaT and propagate to NaN
# in the delivery_days subtraction below.
df_seg[COL_SHIP_DATE] = pd.to_datetime(df_seg[COL_SHIP_DATE], errors = "coerce")
df_seg[COL_ACT_DELIV] = pd.to_datetime(df_seg[COL_ACT_DELIV], errors = "coerce")

# Delivery days: actual transit time from shipment to delivery.
# This measures the network's physical delivery performance, independent
# of SLA thresholds. Global Logistics established that this distribution
# is consistent across corridors (median 3 days, P90 6 days) - the capped
# version will be used to confirm this holds across customer segments.
df_seg["delivery_days"] = (df_seg[COL_ACT_DELIV] - df_seg[COL_SHIP_DATE]).dt.days

# Clip negative values to zero.
# Negative lead times indicate data entry errors (delivery recorded before
# shipment) or backdated orders. Capping at zero preserves these records
# for analysis rather than dropping them, while preventing them from
# distorting segment-level means. 
df_seg["delivery_days"] = df_seg["delivery_days"].clip(lower = 0)

# Cap at 99.5th percentile to stabilize distribution tails.
# Extreme outliers (e.g., 30+ day deliveries from returns processing or
# multi-leg exceptions) are real but rare. Without capping, a handful of
# observations can inflate segment-level means and mask the central
# distribution that defines the typical customer experience.
cap = df_seg["delivery_days"].quantile(0.995)
df_seg["delivery_days_capped"] = df_seg["delivery_days"].clip(upper = cap)

# Confirm cap threshold and check distribution shape.
# The describe() output validates that capping affected only the extreme
# right tail (max should equal cap) while leaving the median and quartiles
# unchanged - preserving the core distribution for segment comparisons.
print("Lead-time cap (99.5th pct):", cap)
df_seg[["delivery_days", "delivery_days_capped"]].describe()

Lead-time cap (99.5th pct): 6.0


,delivery_days,delivery_days_capped
count,159763.000000,159763.000000
mean,3.499690,3.499690
std,1.624563,1.624563
min,0.000000,0.000000
25%,2.000000,2.000000
50%,3.000000,3.000000
75%,5.000000,5.000000
max,6.000000,6.000000


In [ ]:
# ----- SERVICE TIER MAPPING -----
# Translate RFM segments into operational service tiers.
# Each tier implies a distinct fulfillment posture:
#   Priority Protection   -> Mode reassignment, SLA calibration, proactive monitoring
#   Growth Priority       -> Mode review, service consistency protection
#   Standard Service      -> Cost-optimized modes, no premium investment
#   Recovery/Low Priority -> Standardized fulfillment, reactivation assessment
tier_map = {
    "High-Value At-Risk": "Priority Protection",
    "Loyal/Active": "Growth Priority",
    "Others": "Standard Service",
    "Dormant/At-Risk": "Recovery/Low Priority"
}

In [ ]:
# -------------- LEAD TIME DISTRIBUTION BY SEGMENT AND TIER ----------------------

# Segment-level service diagnostics.
# This table combines actual delivery performance (lead time), observed
# service outcomes (late rate), predicted risk (avg risk score), and
# commercial weight (sales, sales share) for each RFM segment. It directly
# answers the service equity question: do high-value segments experience
# different delivery physics or different SLA outcomes than the rest of the base?
#
# Global Logistics established that physical lead times are consistent
# across the network. If this table shows similar p50/p90 delivery days
# across segments but divergent late rates, it confirms that the gap is
# SLA-driven - not a delivery speed problem - and that the divergence
# tracks with shipping mode assignment differences across segments.
seg_lead = df_seg.groupby("segment").agg(
    customers = (COL_CUSTOMER_ID, "nunique"),
        # Unique customers per segment - weights strategic reach
    orders = (COL_ORDER_ID, "nunique"),
        # Order count per segment - weights operational significance
    avg_delivery_days = ("delivery_days_capped", "mean"),
        # Mean capped lead time - central tendency of delivery speed
    p50_delivery_days = ("delivery_days_capped", lambda s: s.quantile(0.5)),
        # Median lead time - expected ~3 days per Global Logistics
    p90_delivery_days = ("delivery_days_capped", lambda s: s.quantile(0.9)),
        # 90th percentile - expected ~6 days per Global Logistics
    late_rate = (COL_LATE_RISK, "mean"),
        # Observed late delivery proportion - the service outcome
    avg_risk = (COL_RISK_SCORE, "mean"),
        # Mean predicted risk score - forward-looking exposure estimate
    sales = ("sales_num", "sum")
        # Total sales per segment - commercial weight for prioritization
).reset_index().sort_values("orders", ascending = False)

# Sales share - what proportion of total revenue each segment represents.
# This directly links service performance to commercial exposure.
seg_lead["sales_share"] = seg_lead["sales"] / seg_lead["sales"].sum()

# Map segments to operational service tiers.
# Translates descriptive classification into prescriptive fulfillment posture.
seg_lead["service_tier"] = seg_lead["segment"].map(tier_map)

seg_lead

,segment,customers,orders,avg_delivery_days,p50_delivery_days,p90_delivery_days,late_rate,avg_risk,sales,sales_share,service_tier
1,High-Value At-Risk,7562,34453,3.505407,3.0,6.0,0.547772,0.548716,1.608705e+07,0.553726,Priority Protection
3,Others,11951,30160,3.493832,3.0,6.0,0.550918,0.549924,1.264689e+07,0.435313,Standard Service
2,Loyal/Active,623,623,3.593900,3.0,6.0,0.566613,0.560583,8.229662e+04,0.002833,Growth Priority
0,Dormant/At-Risk,516,516,3.356061,3.0,6.0,0.509091,0.517851,2.361502e+05,0.008128,Recovery/Low Priority


In [ ]:
# ----- TAIL EXPOSURE -----
# Tail exposure measures the proportion of orders that meet or exceed the
# 90th percentile of delivery days. These extreme delays drive escalations,
# refunds, and churn risk disproportionate to their frequency.
# Given Global Logistics established that P90 is uniform across the network
# (6 days), segment-level tail exposure is expected to cluster around the
# global rate. This calculation provides the baseline before confirming
# that no segment is disproportionately exposed.

# Define tail threshold and flag individual orders
tail_threshold = df_seg["delivery_days_capped"].quantile(0.90)
df_seg["is_tail_delay"] = (df_seg["delivery_days_capped"] >= tail_threshold).astype(int)

# Tail threshold - expected value is 6 days per Global Logistics P90
print("Tail threshold (p90 of capped delivery_days):", tail_threshold)
print("Overall tail exposure rate:", round(df_seg["is_tail_delay"].mean(), 3))

Tail threshold (p90 of capped delivery_days): 6.0
Overall tail exposure rate: 0.16


### Lead Time Distribution by Customer Segment and Value Tier

**(SCOR: Deliver / Return)**

This analysis evaluates how delivery performance distributes across customer segments. It integrates RFM-derived segmentation with engineered lead time metrics and late delivery risk scores to answer a single question: is service performance aligned to customer value, or is it uniform regardless of value contribution?

|Segment |Orders |Avg Delivery Days |P50 |P90 |Late Rate |Avg Risk|
|--------|-------|------------------|----|----|----------|--------|
|High-Value At-Risk |34,453 |3.51 |3 |6 |54.8% |0.549|
|Others |30,160 |3.49 |3 |6 |55.1% |0.550|
|Loyal/Active |623 |3.59 |3 |6 |56.7% |0.561|
|Dormant/At-Risk |516 |3.36 |3 |6 |50.9% |0.518|

#### 1. Structural Service Uniformity

Delivery performance is effectively identical across all segments.

- Median delivery time: 3 days across every cohort.
- P90 tail exposure: 6 days across every cohort.
- Average delivery days range from 3.36 to 3.59, a spread of less than a quarter-day.

This confirms the Global Logistics finding: the fulfillment network delivers consistent physical performance. Lead times do not vary meaningfully by customer segment. The network is operationally uniform.

But it also reveals something Global Logistics could not: that uniformity is the problem. High-value customers are not receiving materially faster or more reliable delivery than lower-value cohorts. The network operates on volume-driven logistics, not value-driven logistics. Service is standardized when it should be stratified.

This uniformity reflects near-identical shipping mode allocation across segments: approximately 60% Standard Class, 20% Second Class, 15% First Class, and 5% Same Day regardless of customer value.

#### 2. High-Value At-Risk: Revenue Concentration Without Service Differentiation

This segment drives the largest shipment volume - over 34,000 orders. Prior analysis established that it concentrates profit contribution, drives demand repeatability, and carries disproportionate revenue-at-risk.

Yet operationally, it receives no differentiated treatment:

- Average delivery days: 3.51 (global mean).
- Late rate: 54.8% (global mean).
- Risk score: 0.549 (global mean).

The supply chain is economically dependent on this cohort but does not service it differently. The result is structural exposure: late deliveries here impact the orders with the highest lifetime value at stake. The fix is mode reassignment, moving these customers out of Second Class and First Class shipping into modes whose SLA thresholds match the delivery physics the network already provides.

#### 3. Loyal/Active: Service Sensitivity in the Growth Core

This segment is small in volume (623 orders) but shows the highest late rate (56.7%) and elevated risk score (0.561).

These customers are recently active, frequent purchasers, and behaviorally engaged. Yet they experience service outcomes marginally worse than the baseline population. The difference is small in absolute terms, but it matters because of who these customers are: the proven repeat purchasers whose retention stabilizes Plan forecasts.

Their elevated late rate, like all segments, is mode-driven. If Loyal/Active customers are disproportionately assigned to Second Class or First Class shipping, their worse outcome is an SLA artifact, not a delivery failure. Mode review for this segment protects the growth pipeline.

#### 4. Dormant/At-Risk: Lower Volume, Lower Exposure

This cohort records the lowest average delivery days (3.36), lowest late rate (50.9%), and lowest risk exposure overall.

The result appears counterintuitive - why would at-risk customers show better service metrics? 

Two structural explanations apply:

- **Reduced interaction volume.** Fewer shipments mean fewer opportunities for late deliveries to register.
- **Engagement decay.** Customers who have already disengaged may no longer be placing orders that use high-late shipping modes. Their improved metrics reflect reduced logistical interaction, not improved service quality.

The relevant service history for this segment is in the past - the deliveries that occurred when they were active. Current metrics measure residual activity, not the experience that may have driven disengagement.

#### 5. Tail Risk: A Universal Structural Feature

P90 delivery is 6 days across every segment, with approximately 16% of orders falling into the tail. These tail delays are structurally embedded in the network; exposure is uniform, meaning no cohort is insulated from or disproportionately exposed to extreme delays.

This uniformity of tail exposure reinforces the Global Logistics conclusion that late delivery drivers are systemic and mode-linked, not segment-specific. Tail risk mitigation - where warranted - is a network-level or mode-level intervention, not a segment-level one. The question is whether the business cost of reducing P90 from 6 days justifies the investment, given that late delivery penalties are 13.7% of reported profit and discount leakage is the dominant margin threat.

#### 6. Cross-Notebook Integration

This segment-level service view connects the full analytical chain:

- **Global Logistics** established that physical lead times are consistent and late rates diverge by shipping mode SLA design. This table confirms both findings hold at the customer segment level.

- **Financial Impact** showed that late delivery penalties ($476.9K) are secondary to discount leakage ($3.26M) and returns are negligible ($73.1K). This table shows that the segments bearing those penalties - High-Value At-Risk and Others - receive no service prioritization that would reduce them.

- **Operational Risk** predicted late probability by mode. This table shows those predictions materialize uniformly across segments, confirming that mode assignment is the transmission mechanism.

This table is the connective layer between logistics performance, customer experience, and financial outcomes.

#### 7. Strategic Output: Service Allocation Misalignment

The central finding is structural: the network operates on standardized fulfillment, not value-differentiated fulfillment.

High-value customers absorb the same SLA-driven late rates as the rest of the base. Growth cohorts face the same delivery experience as dormant accounts. Service investment is not targeted by customer value.

The correction is not operational process redesign. The network delivers consistently. The correction is mode reassignment and SLA calibration aligned to customer value:   

- **High-Value At-Risk:** Reassign from Second Class and First Class to Standard Class shipping, where SLA thresholds match delivery physics and late rates are approximately half.   
- **Loyal/Active:** Review mode allocation to ensure growth-cohort customers are not exposed to avoidable SLA-driven late deliveries.   
- **Others / Dormant:** Maintain standardized, cost-optimized modes.

#### Key Takeaways

Delivery performance, while operationally consistent, is not strategically differentiated. All customer segments experience the same lead times, the same tail risk, and the same late rates, regardless of their value contribution or future revenue potential.

This creates a structural misalignment: the customers who matter most economically are not the customers most protected operationally. Correcting this does not require the network to deliver faster. It requires the service promises attached to each customer's shipping mode to reflect what the network already does, and for high-value customers to be assigned to the modes where those promises are already calibrated correctly. That is a service design decision, not a logistics investment, and it represents one of the highest-return levers available for protecting customer lifetime value through fulfillment design.

These segments translate directly into operational service tiers:   

- High-Value At-Risk → Priority Protection (mode reassignment). 
- Loyal/Active → Growth Priority (mode review). 
- Others → Standard Service (cost-optimized).
- Dormant/At-Risk → Recovery/Low Priority (efficient fulfillment).

## Repeat Purchasing Proxy

**(SCOR: Plan ↔ Deliver)**

The dataset lacks an explicit churn label, making it impossible to prove causation between service failure and customer loss. But a directional test is possible: *do customers who experience late deliveries show weaker repeat purchasing signals?*

Repeat purchasing is proxied using two observable metrics:

- Order frequency: Total orders per customer - does it differ by late delivery exposure?
- Post-late purchasing rate: Whether a customer places another order after experiencing a late delivery.

This is not causal proof. A customer may stop purchasing for reasons unrelated to delivery performance. But if the data shows a consistent negative association between late exposure and repeat purchasing, it provides operational evidence that service reliability affects demand stability - linking Deliver performance directly to Plan forecasting confidence.

In [ ]:
# --------------- POST-LATE REPURCHASE PROXY --------------

# Customer-level late exposure summary.
# Aggregates each customer's full order history into a single row capturing
# whether they ever experienced a late delivery, their overall late rate,
# total orders, total spend, and most recent purchase date.
# This table is the foundation for testing whether late exposure correlates
# with weaker repeat purchasing behavior.
cust_late_exposure = df_seg.groupby(COL_CUSTOMER_ID).agg(
    # Binary: 1 if the customer experienced at least one late delivery, 0 otherwise
    any_late = (COL_LATE_RISK, "max"),
    # Proportion of orders delivered late - intensity of exposure    
    late_rate = (COL_LATE_RISK, "mean"),
    # Total order count - the repeat purchasing proxy    
    orders = (COL_ORDER_ID, "nunique"),
    # Total spend - value context for the frequency comparison    
    monetary = ("sales_num", "sum"),
    # Most recent order - enables recency-filtered sub-analysis if needed    
    last_order_date = (COL_ORDER_DATE, "max")
).reset_index()

# Compare repeat purchasing behavior: customers with vs without late exposure.
# If late deliveries correlate with reduced engagement, the "Had Late Exposure"
# group should show lower avg_orders and median_orders.
# Important caveat: this is directional evidence, not causal proof.
# Customers with more orders have more opportunities to experience a late
# delivery, so the relationship runs in both directions. The median mitigates
# this somewhat by reducing the influence of high-frequency outliers.
summary_late_vs_not = cust_late_exposure.groupby("any_late").agg(
    # Customer count per group - confirms base sizes are sufficient for comparison
    customers = (COL_CUSTOMER_ID, "nunique"),
    # Mean order count - sensitive to high-frequency customers    
    avg_orders = ("orders", "mean"),
    # Median order count - more robust given the right-skewed frequency distribution    
    median_orders = ("orders", "median"),
    # Mean spend - provides value context alongside frequency   
    avg_monetary = ("monetary", "mean"),
    # Median spend - robust to VIP outliers    
    median_monetary = ("monetary", "median"),
        
).reset_index()

# Map binary labels
summary_late_vs_not["any_late"] = summary_late_vs_not["any_late"].map({
    0: "No Late Exposure",
    1: "Had Late Exposure"
})

summary_late_vs_not

,any_late,customers,avg_orders,median_orders,avg_monetary,median_monetary
0,No Late Exposure,4634,1.272767,1.0,436.086951,252.880005
1,Had Late Exposure,16018,3.736671,4.0,1687.573744,1568.050020


In [ ]:
#  ----- RELATIONSHIP STRENGTH: SERVICE VS REPEAT PROXY -----

# These correlations test for a linear relationship between late delivery
# exposure and repeat purchasing behavior at the customer level.
# A meaningful negative coefficient would suggest service failures suppress
# engagement. A near-zero coefficient confirms the cohort finding: late
# exposure tracks with transaction volume (more orders → more late events),
# not with behavioral disengagement (late events → fewer orders).

# Late rate vs order count
# Expected: ~0. Confirms late exposure is a mechanical byproduct of frequency,
# not a driver of reduced purchasing.
corr_orders = cust_late_exposure["late_rate"].corr(cust_late_exposure["orders"])

# Late rate vs monetary value
# Expected: ~0. Confirms the same pattern holds for spend - high-value
# customers are not spending less because of late deliveries.
corr_monetary = cust_late_exposure["late_rate"].corr(cust_late_exposure["monetary"])

print("Correlation: customer late_rate vs order count:", round(corr_orders, 3))
print("Correlation: customer late_rate vs monetary:", round(corr_monetary, 3))

Correlation: customer late_rate vs order count: -0.002
Correlation: customer late_rate vs monetary: -0.002


### Post-Late Repurchase Proxy Analysis

**(SCOR: Deliver / Return / Plan)**

This analysis tests whether late delivery exposure correlates with reduced repeat purchasing. The dataset lacks explicit churn or satisfaction signals, so repurchase activity serves as a directional proxy for customer retention following service disruption.

#### 1. Cohort Definitions

Customers are split by exposure:

|Cohort |Definition|
|-------|----------|
|No Late Exposure |Zero late deliveries across all orders|
|Had Late Exposure |At least one late delivery|

#### 2. Repurchase Intensity

|Metric |No Late Exposure |Had Late Exposure|
|-------|-----------------|-----------------|
|Customers |4634 | 16018 |
|Average Orders |1.27 |3.74|
|Median Orders |1 |4|

The exposed cohort places more orders. This is not evidence that late deliveries drive repurchase. It reflects a mechanical relationship: customers who place more orders have more opportunities to experience a late delivery. The median (4 vs 1) confirms this is not an artifact of a few outliers, the typical exposed customer is a higher-frequency buyer.

Customer-level correlation between late rate and order count is -0.002 - effectively zero - confirming that the relationship is structural (more orders → more exposure), not behavioral (late deliveries → reduced purchasing).

#### 3. Monetary Value

|Metric |No Late Exposure |Had Late Exposure|
|-------|-----------------|-----------------|
|Average Monetary ($)|436 |1,688|
|Median Monetary ($)|253 |1,568|

Customers with late exposure generate approximately 4X the revenue of those without. This aligns with the segment and value band analyses: late deliveries disproportionately reach the highest-value customers because those customers transact more.

#### 4. Behavioral Resilience, Not Causal Reassurance

The exposed cohort continues purchasing - median 4 orders, median spend ~$1,568. This indicates that service failures, at the observed frequency and severity, have not triggered widespread disengagement. Switching costs, brand loyalty, or lack of alternatives may explain the persistence.

But this resilience is not immunity. The dataset cannot measure whether these customers would purchase more in the absence of late deliveries, or whether satisfaction is silently degrading. What it can measure - and what the Financial Impact notebook established - is that late deliveries impose a direct cost: $476.9K in penalties, 13.7% of reported profit.

#### 5. The Finding, Calibrated

This proxy analysis reframes late delivery impact in two ways:

1. **Late exposure tracks with customer value, not customer churn.** The customers most affected are the ones who buy most frequently and spend the most.
2. **The primary observable consequence is profitability erosion, not revenue loss.** Customers keep buying. But each late delivery extracts a penalty, and those penalties concentrate in the highest-value accounts.

Late delivery is less a demand risk than a margin risk.

#### Cross-Notebook Integration

- **Demand Forecasting** identified where revenue concentrates.
- **Financial Impact** quantified where profit leaks - $3.26M to discounts, $476.9K to late penalties.
- **Global Logistics** identified the mechanism: mode-SLA mismatch, not delivery capability.

This analysis identifies who absorbs the impact: high-frequency, high-value customers who continue purchasing but generate disproportionate penalty costs.

#### Strategic Output

The mandate is not to prevent revenue collapse, revenue is holding. The mandate is to protect margin in the accounts that matter most, by addressing the mode-SLA mismatch that generates the penalties.

#### Key Takeaways

Customers who experience late deliveries are not disengaging. They are the most commercially important accounts in the network.

This creates a specific strategic posture: protect high-frequency customers from SLA-driven late deliveries - not because revenue is at immediate risk, but because margin, loyalty, and lifetime value are being eroded silently in the accounts the business can least afford to damage. The intervention remains mode reassignment and SLA calibration, applied to the customers who generate the most exposure.

## Strategic Close: Customer-Centric Operations

**(SCOR: Plan / Deliver / Return / Measure)**

This notebook operationalizes customer experience as a measurable, economically governed supply chain capability. By integrating behavioral segmentation, operational service performance, and fulfillment risk exposure, customer operations move from a reactive service function to a proactive strategic lever.

### Four structural findings emerge:

1. **Customer value is highly concentrated.** The High-Value At-Risk segment drives over 55% of revenue. Protecting these relationships is not a service ambition, it is a financial requirement. Shipping mode allocation and SLA calibration must safeguard this value base.

2. **Service performance is uniform across segments, and that is the problem.** Median lead time is 3 days and late rates cluster around 55% regardless of customer value. The network delivers consistently, but it does not differentiate. High-value customers receive the same SLA-driven late exposure as the rest of the base because shipping mode allocation is nearly identical across segments.

3. **The root cause is mode-SLA mismatch, not logistics capability.** Global Logistics established that late rates diverge by shipping mode, not by geography or corridor. Second Class (~76% late) and First Class (~95% late) carry structurally high failure rates because their SLA thresholds do not match the delivery physics the network consistently provides. Standard Class (~38% late) is the baseline reliability benchmark. The intervention is mode reassignment and SLA recalibration, a service design decision, not a logistics investment.

4. **Late delivery is a margin risk, not a revenue risk.** Customers who experience late deliveries are the most frequent, highest-spending accounts in the network. They continue purchasing - repeat purchasing shows no negative correlation with late exposure. But each late delivery extracts a financial penalty ($476.9K, 13.7% of reported profit), and those penalties concentrate in the highest-value accounts. The Financial Impact notebook established that discount leakage ($3.26M) is the dominant margin threat, but late delivery penalties are an avoidable secondary drain on the accounts that matter most.

**These findings translate into a differentiated operating model:**

- **Priority Protection** for High-Value At-Risk customers: mode reassignment from Second Class and First Class to Standard Class, where SLA thresholds match delivery physics.
- **Growth Priority** for Loyal/Active customers: mode allocation review to protect the repeat purchasing core.
- **Standard Service** for the Others baseline: cost-optimized, standardized fulfillment.
- **Recovery/Low Priority** for Dormant accounts: efficient modes, selective reactivation.

### Cross-notebook integration:

- **Demand Forecasting** identified where revenue will emerge.
- **Financial Impact** quantified where profit leaks and that discounting, not late delivery, is the primary threat.
- **Operational Risk** predicted where service failures would occur.
- **Global Logistics** identified the mechanism: mode-SLA design, not logistics capability.

Customer-Centric Operations determines who absorbs the impact and where intervention yields the highest retention and margin return.

The result is a service architecture that does not require the network to move faster. It requires the right customers to be assigned to the right shipping modes, the ones whose promises the network already keeps. That is a customer-intelligent, profit-aligned supply chain design.

## Outputs for Power BI

Two tables are exported to support flexible reporting in Power BI - the enriched order-level table serves as a self-contained analytical foundation, while the customer RFM table provides a dedicated customer dimension for the BI export pipeline.

`df_seg` **(customer_centric_ops.parquet):** Transaction-level data with all customer attributes attached - segment labels, RFM scores, value bands, service tier assignments, late exposure metrics, lead time, and tail delay flags. Power BI can aggregate to customer-level or drill to order-level from this single table, enabling visuals such as:

- Service performance (late rate, lead time) by customer segment and value band.
- Late delivery exposure by shipping mode and service tier.
- Revenue concentration vs. service reliability scatterplots.
- Retention-risk monitoring: high-value customers with increasing recency and elevated late exposure.

`cust` **(customer_segments_rfm.parquet):** Customer-level RFM table with segment and service tier mappings. This table serves as a dedicated customer dimension within the BI export pipeline, where it is joined with other business views to produce the final Power BI source files. Separating it from the enriched orders table avoids dimensional redundancy.

In [12]:
# Map segments to operational service tiers in the customer RFM table.
cust["service_tier"] = cust["segment"].map(tier_map)
df_seg["service_tier"] = df_seg["segment"].map(tier_map)

# Export enriched order-level table.
trans_seg_file_path = "customer_centric_ops.parquet"
df_seg.to_parquet(trans_seg_file_path, index=False)

# Export customer RFM table.
cust_rfm_file_path = "customer_segments_rfm.parquet"
cust.to_parquet(cust_rfm_file_path, index=False)

print(f"File saved successfully: {trans_seg_file_path}")
print(f"File saved successfully: {cust_rfm_file_path}")

File saved successfully: customer_centric_ops.parquet
File saved successfully: customer_segments_rfm.parquet
